##### Loading the data

In [1]:
# Use the same ground truth questions:
import pandas as pd

df_ground_truth = pd.read_csv("ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
%pip install minsearch

In [3]:
# Load the FAQ documents and the search index:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
# Create a lookup table:
doc_idx = {}

for doc in documents:
    doc_idx[doc["doc_id"]] = doc

##### Running the agent

In [ ]:
import os

In [ ]:
from openai import OpenAI
from toyaikit.llm import OpenAIClient

# Resolve API key from environment if available (OPENAI_API_KEY preferred, fallback to GROQ_API_KEY)
api_key = os.getenv("OPENAI_API_KEY") or os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY or GROQ_API_KEY environment variable before creating OpenAI client.")
openai_client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1") or os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")  
)

In [15]:
# Define the search tool:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [13]:
# Create the runner:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="openai/gpt-oss-20b", client=openai_client)
)

In [17]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [19]:
# Run it for one ground truth question:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

/usr/local/lib/python3.12/dist-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-20b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [20]:
result

LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None), EasyInputMessage(content="Can I enroll in the course now that I've found it?", role='user', phase=None, type=None), ResponseReasoningItem(id='resp_01kxkacg3bfqwsn57692w330jb', summary=[], type='reasoning', content=[Content(text='We need to answer question: "Can I enroll in the course now that I\'ve found it?" Need to search FAQ database. Use search.', type='reasoning_text')], encrypted_content=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"enroll in the course now that I\'ve found it"}', call_id='fc_4c632687-358c-4589-9b76-046171804fa5', name='search', type='function_call', id='fc_4c632687-358c-4589-9b76-046171804fa5', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'fc_4c632687-358c-4589-9b76-046171804f

In [21]:
# Look at the full message history:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content="Can I enroll in the course now that I've found it?", role='user', phase=None, type=None),
 ResponseReasoningItem(id='resp_01kxkacg3bfqwsn57692w330jb', summary=[], type='reasoning', content=[Content(text='We need to answer question: "Can I enroll in the course now that I\'ve found it?" Need to search FAQ database. Use search.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"enroll in the course now that I\'ve found it"}', call_id='fc_4c632687-358c-4589-9b76-046171804fa5', name='search', type='function_call', id='fc_4c632687-358c-4589-9b76-046171804fa5', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'fc_4c632687-358c-4589-9b76-046171804fa5',
  'output': '

In [22]:
# Extract the function name and arguments:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [ ]:
# For this example:
tool_calls = extract_tool_calls(result.all_messages)
tool_calls

[{'name': 'search',
  'arguments': '{"query":"enroll in the course now that I\'ve found it"}'}]

In [26]:
# Get the original answer:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [28]:
# Save the A->Q->A' record and the trajectory:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost,
    "document": doc_id,
}

agent_result

{'question': "Can I enroll in the course now that I've found it?",
 'answer_agent': 'Yes!\u202fYou can enroll and start the course at any time. The LLM Zoomcamp is open‑access, so once you locate the course (e.g., through the course website or the Zoomcamp portal) you can simply register and begin working through the materials.\n\n**What to do next**\n\n1. **Register** – Sign up on the course platform (or GitHub) if you haven’t already.  \n2. **Check the schedule** – Deadlines and weekly workflow are listed on the course platform.  \n3. **Get the resources** – Download the videos, notebooks, and project repo from the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/) and the [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).  \n4. **Follow the typical workflow** –  \n   - Watch the lesson videos.  \n   - Work through the accompanying notebooks/code.  \n   - Review homework instructions on GitHub.  \n   - Submit your solutions before the deadline via

#### Processing multiple questions

In [37]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost,
        "document": doc_id,
    }

    return answer_record

In [40]:
# Run it for a small sample in parallel:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:3], generate_agent_answer)


  0%|          | 0/3 [00:00<?, ?it/s]

el: {'question': "Can I enroll in the course now that I've found it?", 'document': '74eb249bbf'}
el: {'question': 'What do I need to do to earn a certificate if I join late?', 'document': '74eb249bbf'}
el: {'question': 'Is there still time to submit my project for a certificate?', 'document': '74eb249bbf'}


In [41]:
# Turn it into a dataframe:
df_agent = pd.DataFrame(agent_answers)

In [42]:
# Calculate the total cost:
df_agent["cost"].sum()

0

In [43]:
# Save the results:
df_agent.to_csv("agent-answers.csv", index=False)